In [1]:
# imports
from src.utils import load_env, load_json, get_logger
from src.experiment_config import LLMExperimentConfig
from src.data_loading import DatasetLoader

/Users/alexleto/projects/metaphor-detector/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env_vars = load_env()
logger = get_logger("check_datasets")

DATASET = "tweets_immigration_labeled"
CANDIDATE_PATH = f"{env_vars["RESULTS_DIR"]}/get_candidate_metaphors/rule_based"
MET_CLASS_PATH = f"{env_vars["RESULTS_DIR"]}/metaphor_classification/llm"

In [3]:
# load the candidate metaphors
candidate_data = load_json(f"{CANDIDATE_PATH}/{DATASET}_metaphor_paths.json")
candidate_config = candidate_data["config"]
candidate_data = candidate_data["data"]

can_doc_ids = [c_id.split("_")[0] for c_id in candidate_data.keys()]
can_doc_ids = list(set(can_doc_ids))

In [4]:
met_classifications = load_json(f"{MET_CLASS_PATH}/{DATASET}_source_verb_target_noun_binary_met_class_qwen.json")
met_class_config = met_classifications["config"]
met_classifications = met_classifications["data"]
class_doc_ids = [c_id.split("_")[0] for c_id in met_classifications.keys()]
class_doc_ids = list(set(class_doc_ids))


In [5]:
for class_k, class_v in met_classifications.items():
    if class_k not in candidate_data.keys():
        print(f"{class_k} not in candidates")

In [6]:
# print some stats
print(DATASET)
print(f"    documents: {len(can_doc_ids)}")
print(f"    candidate paths: {len(candidate_data)}")
print(f"    avg. candidates per doc: {len(candidate_data)/len(can_doc_ids)}")
print(f"    llm-processed documents: {len(class_doc_ids)}")
print(f"    llm-processed candidates: {len(met_classifications)}")
print(f"    avg. processed candidates per doc: {len(met_classifications)/len(class_doc_ids)}")

tweets_immigration_labeled
    documents: 1489
    candidate paths: 7042
    avg. candidates per doc: 4.7293485560779045
    llm-processed documents: 1489
    llm-processed candidates: 7042
    avg. processed candidates per doc: 4.7293485560779045


In [7]:
print(env_vars)

{'REPO_PATH': '/Users/alexleto/projects/metaphor-detector', 'RAW_DATA_DIR': '/Users/alexleto/projects/metaphor-detector/data/raw', 'DB_DIR': '/Users/alexleto/projects/metaphor-detector/data/dbs', 'INTERIM_DIR': None, 'RESULTS_DIR': '/Users/alexleto/projects/metaphor-detector/results', 'RANDOM_SEED': 42, 'DEVICE': 'cuda'}


In [8]:
config = LLMExperimentConfig.from_dict(met_class_config, logger, env_vars)
data_loader = DatasetLoader(config)

2026-05-13 12:33:10,905 - check_datasets - INFO - Loading annotation config from /Users/alexleto/projects/metaphor-detector/src/llm_ann/configs/qwen3.yaml


In [9]:
# data = data_loader.load_raw_data(processed_only=True)
data = data_loader.load_raw_data()
assert len(data) == len(class_doc_ids), f"Number of data points in df ({len(data)}) != number of docs in LLM labeled met data ({len(class_doc_ids)})"

AssertionError: Number of data points in df (1596) != number of docs in LLM labeled met data (1489)

In [10]:
# check that the dataset is balanced
grouped_data = data.groupby(by="polarity").count()
grouped_data

,Unnamed: 0,id_str,concept,No,Yes,total,percent_no,percent_yes,text,id,ideology
polarity,,,,,,,,,,,
left,1037,1037,1037,1037,1037,1037,1037,1037,1037,1037,1037
right,559,559,559,559,559,559,559,559,559,559,559
